# Prediction Model Tutorial

The purpose of this notebook is to demonstrate the difference between training and prediction models in the context of recurrent neural networks.

## Setup

In [ ]:
import xarray as xr
import rioxarray as rxr
from pyproj import Transformer
import sys
sys.path.append("..")
import tensorflow as tf
import numpy as np
import glob
import re
import os.path as osp
import pandas as pd
from datetime import datetime
import matplotlib.pyplot as plt
from utils import retrieve_url, str2time, read_pkl, read_yml
from data_funcs import int2fstep
from moisture_rnn import RNNParams
from moisture_rnn_xarray import bands_to_names, names_to_bands, get_file_list, preprocess, calc_eqs, calc_rain, bbox_to_xy

## User Options

In [ ]:
start_time = 2024042000            # start time for predictions (YYYMMDDHH)
end_time = 2024042002              # end time for predictions (YYYMMDDHH)
forecast_step = 3                  # forecast step for HRRR model
base_url = "https://demo.openwfm.org/web/data/fmda/tif/" # base URL for staged data
bbox = [37, -111, 46, -95]         # Spatial bounding box for preds (optional)
data_path = "../data"

## Create Models

In [ ]:
# Training Data Object
rnn_dat = read_pkl("../outputs/models/rnn_data_rocky.pkl")
# Params
params = read_yml("../params.yaml", subkey="rnn")

In [ ]:
params.update({
    'hidden_layers': ['lstm', 'dense'],
    'hidden_units': [32, 16],
    'hidden_activation': ['tanh', 'relu'],
    'batch_size': 64,
    'return_sequences': False,
    'features_list': ['Ed', 'Ew', 'rain']
})
params = RNNParams(params)

In [ ]:
from tensorflow.keras import layers,models


def build_hidden_layers(x, stateful=True, return_sequences=True):
    # params = self.params
    last_recurrent = None
    
    # Identify the last RNN/LSTM layer, unless an Attention layer follows it
    for i, layer_type in enumerate(params['hidden_layers']):
        if layer_type in ['rnn', 'lstm']:
            # Check if there's an Attention layer following the current RNN/LSTM layer
            if i < len(params['hidden_layers']) - 1 and params['hidden_layers'][i + 1] == 'attention':
                continue
            last_recurrent = i        
    
    # Loop over each layer specified in 'hidden_layers'
    for i, layer_type in enumerate(params['hidden_layers']):
        units = params['hidden_units'][i]
        activation = params['hidden_activation'][i]

        if layer_type == 'dense':
            x = layers.Dense(units=units, activation=activation)(x)

        elif layer_type == 'dropout':
            x = layers.Dropout(params['dropout'])(x)
        
        elif layer_type == 'rnn':

            print()
            
            is_last_recurrent = (i == last_recurrent)
            return_seqs_logic = not is_last_recurrent or return_sequences
            x = layers.SimpleRNN(units=units, activation=activation, dropout=params['dropout'], recurrent_dropout=params['recurrent_dropout'], stateful=stateful,
                                 return_sequences=return_seqs_logic)(x)
        
        elif layer_type == 'lstm':
            is_last_recurrent = (i == last_recurrent)
            return_seqs_logic = not is_last_recurrent or return_sequences
            x = layers.LSTM(units=units, activation=activation, dropout=params['dropout'], recurrent_dropout=params['recurrent_dropout'], stateful=stateful,
                            return_sequences=return_seqs_logic)(x)    
        
        elif layer_type == 'attention':
            # Self-attention mechanism
            x = layers.Attention()([x, x])
        elif layer_type == 'conv1d':
            kernel_size = params.get('kernel_size', 3)  # Check for kernel size, use 3 if missing
            x = layers.Conv1D(filters=units, kernel_size=kernel_size, activation=activation, padding='same')(x)
        else:
            raise ValueError(f"Unrecognized layer type: {layer_type}, skipping")
    
    return x

In [ ]:
optimizer=tf.keras.optimizers.Adam(learning_rate=params['learning_rate'])

In [ ]:
# Define the input layer with the specified batch size, timesteps, and features
inputs = tf.keras.Input(batch_shape=(params['batch_size'], params['timesteps'], params['n_features']))
x = inputs
# Build hidden layers
x = build_hidden_layers(x, stateful = params['stateful'], return_sequences = params['return_sequences'])    

# Add the output layer
if params['output_layer'] == 'dense':
    outputs = layers.Dense(units=params['output_dimension'], activation=params['output_activation'])(x)
else:
    raise ValueError("Unsupported output layer type: {}".format(params['output_layer']))

# Create the model
model_train = models.Model(inputs=inputs, outputs=outputs)
model_train.compile(loss='mean_squared_error', optimizer=optimizer)

In [ ]:
model_train.summary()

In [ ]:
# Define the input layer with flexible batch size and sequence length
inputs = tf.keras.Input(shape=(None, params['n_features']))
x = inputs
# Build hidden layers
x = build_hidden_layers(x, stateful=False, return_sequences = True)    

# Add the output layer
if params['output_layer'] == 'dense':
    outputs = layers.Dense(units=params['output_dimension'], activation=params['output_activation'])(x)
else:
    raise ValueError("Unsupported output layer type: {}".format(params['output_layer']))

# Create the prediction model
model_predict = models.Model(inputs=inputs, outputs=outputs)
model_predict.compile(loss='mean_squared_error', optimizer=optimizer)

In [ ]:
model_predict.summary()

In [ ]:
# Define the input layer with flexible batch size and sequence length
inputs = tf.keras.Input(shape=(None, None, params['n_features']))
x = inputs
# Build hidden layers
x = build_hidden_layers(x, stateful=False, return_sequences = True)    

# Add the output layer
if params['output_layer'] == 'dense':
    outputs = layers.Dense(units=params['output_dimension'], activation=params['output_activation'])(x)
else:
    raise ValueError("Unsupported output layer type: {}".format(params['output_layer']))

# Create the prediction model
model_predict_grid = models.Model(inputs=inputs, outputs=outputs)
model_predict_grid.compile(loss='mean_squared_error', optimizer=optimizer)

## Reshape Spatial Data and Apply Model

Steps:
* Extract features based on training data
* Reshape and apply data fitted scaler from training data
* 

In [ ]:
rnn_dat.features_list

In [ ]:
data.band

In [ ]:
Xnew = data.band_data.sel(band=rnn_dat.features_list)
print(Xnew.dims)
print(Xnew.shape)
Xnew = Xnew.stack(spacetime=('x', 'y', 'time'))
Xnew = Xnew.transpose('spacetime', 'band')
print(f"Reshaped Data Shape: {Xnew.shape}")

In [ ]:
np.all(Xnew.band == rnn_dat.features_list).values

In [ ]:
print(f"{rnn_dat.scaler.n_features_in_ = }")

In [ ]:
# Apply scaling (resulting in a numpy array)
Xnew_scaled = rnn_dat.scaler.transform(Xnew)

In [ ]:
# Retrieve original dimensions from Xnew
x, y, time, band = data.sizes['x'], data.sizes['y'], data.sizes['time'], len(rnn_dat.features_list)

# Reshape the scaled array back to the shape (x, y, time, band)
X = Xnew_scaled.reshape(x*y, time, band)

print(X.shape)

# Predict
preds = mod.predict(X)

In [ ]:
# Reshape to Grid and Plot
preds_reshaped = preds.reshape(x, y, time, 1)
preds_da = xr.DataArray(
    preds_reshaped,
    dims=('x', 'y', 'time', 'band'),
    coords={'x': data.coords['x'], 'y': data.coords['y'], 'time': data.coords['time'], 'band': ["preds"]}
)

# # Concatenate preds_da with Xnew along the 'band' dimension
data_preds = xr.concat([data.band_data.sel(band=rnn_dat.features_list), preds_da], dim='band')

In [ ]:
t = 0
plt.figure(figsize=(12, 8))
plt.imshow(data_preds.sel(band="preds").isel(time=t))
plt.title(f"FMC Prediction at time {data.time[t].values.astype('M8[ms]').astype(datetime)}")
plt.colorbar(label="FMC (%)")

In [ ]:
t = 1
plt.figure(figsize=(12, 8))
plt.imshow(data_preds.sel(band="preds").isel(time=t))
plt.title(f"FMC Prediction at time {data.time[t].values.astype('M8[ms]').astype(datetime)}")
plt.colorbar(label="FMC (%)")

In [ ]:
t = 2
plt.figure(figsize=(12, 8))
plt.imshow(data_preds.sel(band="preds").isel(time=t))
plt.title(f"FMC Prediction at time {data.time[t].values.astype('M8[ms]').astype(datetime)}")
plt.colorbar(label="FMC (%)")

In [ ]:
vals = data_preds.sel(band="preds").values

In [ ]:
vals.shape

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.animation import PillowWriter

fig, ax = plt.subplots(figsize=(12, 8))
im = ax.imshow(vals[0], vmin=vals.min(), vmax=vals.max())
cbar = fig.colorbar(im, ax=ax, label="FMC (%)")
title = ax.set_title("")

# Update function for animation
def update(t):
    im.set_array(vals[t])  # Update with the next frame
    time_value = data.time[t].values.astype("M8[ms]").astype(datetime)
    title.set_text(f"FMC Prediction at time {time_value}")

# Create the animation
ani = animation.FuncAnimation(fig, update, frames=vals.shape[0], interval=200)

# Save the animation
ani.save("outputs/animation_map.gif", writer=PillowWriter(fps=2))

In [ ]:
from IPython.display import Image
# Image(filename="outputs/animation_map.gif")